# Surrogate Factory — UCFatigue
## Chapter 7. Model Training
Objectives:
- Train one MLPRegressor per fatigue output.
- Save trained models to artifacts folder.

### 0. Workflow initialisation

In [1]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()

2026-06-25 12:25:47 - SurrogateFactoryLogs - INFO - Initializing Workflow...
2026-06-25 12:25:47 - SurrogateFactoryLogs - INFO - Setting up Workflow folders and paths...
2026-06-25 12:25:47 - SurrogateFactoryLogs - INFO - Changing working directory to Data folder: /Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data
2026-06-25 12:25:47 - SurrogateFactoryLogs - INFO - Loading default metadata schema...
2026-06-25 12:25:47 - SurrogateFactoryLogs - INFO - Tracker requested: 'mlflow'
2026-06-25 12:25:47 - SurrogateFactoryLogs - INFO - Setting up MLflow tracking environment...
Setting tracking uri: file:///Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/mlruns
2026-06-25 12:25:47 - SurrogateFactoryLogs - INFO - Adding methods to Catalog from configuration.
2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - Workflow initialization completed successfully.
2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - Resuming workflow job: 'UCFATIGUE_1'
2026-06-25 12:25:48 

### 7. Model Training

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [3]:
workflow.import_metadata(stage_name="SF_7_Model_Training")

2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - Importing metadata for stage: 'SF_7_Model_Training'
2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - Updating metadata for stage 'Model_Training' in memory.


In [4]:
Train_set = workflow.load_data(workflow.config['job_name'] + '_Train_set.csv')
Val_set   = workflow.load_data(workflow.config['job_name'] + '_Val_set.csv')

2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - Loading data from '/Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data/UCFATIGUE_1_Train_set.csv' (Format: csv)
2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - Successfully loaded data shape: (703, 29)
2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - Loading data from '/Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data/UCFATIGUE_1_Val_set.csv' (Format: csv)
2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - Successfully loaded data shape: (87, 29)


In [5]:
from model_training.learn import train
models = train(workflow, Train_set, Val_set)

2026-06-25 12:25:48 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'train'
MLflow run started: MLP  (id: 661c04957afd44f7b380b66bcba25df5)
Iteration 1, loss = 233.29473692
Validation score: -166.578402
Iteration 2, loss = 218.49007408
Validation score: -172.819832
Iteration 3, loss = 195.88313459
Validation score: -137.937771
Iteration 4, loss = 161.87476439
Validation score: -58.028081
Iteration 5, loss = 117.95113309
Validation score: -13.710088
Iteration 6, loss = 73.72085895
Validation score: -20.697642
Iteration 7, loss = 39.55957580
Validation score: -11.478129
Iteration 8, loss = 18.88768119
Validation score: -9.310528
Iteration 9, loss = 8.40819561
Validation score: -7.366208
Iteration 10, loss = 4.53375807
Validation score: -5.176322
Iteration 11, loss = 3.23701274
Validation score: -3.755952
Iteration 12, loss = 2.75926004
Validation score: -3.075612
Iteration 13, loss = 2.42849609
Validation score: -2.591931
Iteration 14, loss = 2.21087514
Validation score: -2.19

#### Training Curves
Loss curve per epoch (MLP) and train score per boosting iteration (GradientBoosting).

In [6]:
%matplotlib inline
import matplotlib.pyplot as plt
import joblib
import ipywidgets as widgets
from IPython.display import display

models_info = workflow.metadata.get_step_data(["metadata", "Model_Training", "Models"])

tab_contents = []
tab_titles   = []

for info in models_info:
    label        = info["label"]
    outputs_list = info["outputs"]
    model        = joblib.load(info["file"])
    out          = widgets.Output()

    with out:
        if hasattr(model, "loss_curve_"):   # MLP
            has_val = getattr(model, "validation_scores_", None) is not None
            fig, axes = plt.subplots(1, 2 if has_val else 1,
                                     figsize=(13 if has_val else 7, 4))
            ax0 = axes[0] if has_val else axes
            ax0.plot(model.loss_curve_, color="steelblue", linewidth=1.2, label="Train loss")
            if model.best_loss_ is not None:
                ax0.axhline(model.best_loss_, color="tomato", linestyle="--", linewidth=0.9,
                            label=f"Best: {model.best_loss_:.5f}")
            ax0.set_xlabel("Epoch")
            ax0.set_ylabel("MSE Loss")
            ax0.set_title(f"Training Loss  (stopped epoch {model.n_iter_} / {model.max_iter})")
            ax0.legend(fontsize=8)
            if has_val:
                axes[1].plot(model.validation_scores_, color="orange", linewidth=1.2, label="Val R²")
                axes[1].set_xlabel("Epoch")
                axes[1].set_ylabel("R² score")
                axes[1].set_title("Validation Score")
                axes[1].legend(fontsize=8)
            plt.tight_layout()
            plt.show()

        elif hasattr(model, "estimators_"):  # MultiOutputRegressor (GB)
            ncols  = 4
            nrows  = -(-len(outputs_list) // ncols)
            n_est  = model.estimators_[0].n_estimators
            fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
            axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
            for i, (est, col) in enumerate(zip(model.estimators_, outputs_list)):
                axes[i].plot(est.train_score_, color="tomato", linewidth=1)
                axes[i].set_title(col, fontsize=9)
                axes[i].set_xlabel("Iteration", fontsize=8)
                axes[i].set_ylabel("MSE", fontsize=8)
            for j in range(len(outputs_list), len(axes)):
                axes[j].set_visible(False)
            fig.suptitle(f"Train Loss per Boosting Iteration  (n_estimators={n_est})", fontsize=11)
            plt.tight_layout()
            plt.show()

    tab_contents.append(out)
    tab_titles.append(label)

tab = widgets.Tab(children=tab_contents)
for i, title in enumerate(tab_titles):
    tab.set_title(i, title)
display(tab)

### Save

In [7]:
workflow.save_metadata()

2026-06-25 12:25:54 - SurrogateFactoryLogs - INFO - Successfully saved workflow metadata to: /Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data/artifacts/metadata_UCFATIGUE_1.json
